In [1]:
from google.colab import files
import pandas as pd
import numpy as np

# ── Upload visits ──────────────────────────────────────────────────────────────
print("Upload CFB visits CSV...")
up1 = files.upload()
df_visits_raw = pd.read_csv(list(up1.keys())[0])
print(f"Visits raw: {df_visits_raw.shape}")
print(df_visits_raw.head(3))

# ── Upload donations ───────────────────────────────────────────────────────────
print("Upload CFB donations CSV...")
up2 = files.upload()
df_don_raw = pd.read_csv(list(up2.keys())[0])
print(f"Donations raw: {df_don_raw.shape}")
print(df_don_raw.head(3))

Upload CFB visits CSV...


Saving Shared Data Set.xlsx - visits.csv to Shared Data Set.xlsx - visits.csv
Visits raw: (36668, 7)
   Visit Date  Client ID  Household ID  Household Size          Recorded At  \
0  2023-05-01    3141677       1794907               2  2023-03-08 15:50:58   
1  2023-05-01    3310799       1897503               1  2023-03-16 13:41:07   
2  2023-05-01    3808057       2182323               5  2023-03-31 16:35:25   

   Quantity (lbs)  Individuals Served  
0            33.0                   2  
1            23.0                   1  
2            32.5                   5  
Upload CFB donations CSV...


Saving Shared Data Set.xlsx - Donations 2023-2024.csv to Shared Data Set.xlsx - Donations 2023-2024.csv
Donations raw: (922, 3)
           Timestamp     Date   \
0   5/1/2023 9:13:02  5/1/2023   
1  5/1/2023 10:32:54  5/1/2023   
2  5/1/2023 14:18:11  5/1/2023   

   Weight in pounds / lbs (Please check weight is in correct units!)  
0                                               22.0                  
1                                               13.0                  
2                                              450.0                  


In [2]:
# ── Clean visits ───────────────────────────────────────────────────────────────
df_v = df_visits_raw.copy()
df_v["Visit Date"] = pd.to_datetime(df_v["Visit Date"])
df_v["month"] = df_v["Visit Date"].dt.to_period("M").dt.to_timestamp()
df_v["Quantity (lbs)"] = pd.to_numeric(df_v["Quantity (lbs)"], errors="coerce")
df_v["Individuals Served"] = pd.to_numeric(df_v["Individuals Served"], errors="coerce")

# Monthly aggregation
monthly_visits = df_v.groupby("month").agg(
    visits          = ("Client ID",        "count"),
    unique_clients  = ("Client ID",        "nunique"),
    unique_hh       = ("Household ID",     "nunique"),
    people_served   = ("Individuals Served","sum"),
    food_lbs        = ("Quantity (lbs)",   "sum"),
    avg_hh_size     = ("Household Size",   "mean"),
).reset_index()

monthly_visits["lbs_per_person"] = (
    monthly_visits["food_lbs"] / monthly_visits["people_served"].clip(lower=1)
)

print(f"\nMonthly visits: {len(monthly_visits)} months")
print(f"Date range: {monthly_visits['month'].min().date()} → {monthly_visits['month'].max().date()}")
print(monthly_visits.head(6))

# ── Clean donations ────────────────────────────────────────────────────────────
df_d = df_don_raw.copy()

# Handle the date column — yours is called "Date " (with trailing space possibly)
date_col = [c for c in df_d.columns if "date" in c.lower()][0]
df_d["date"] = pd.to_datetime(df_d[date_col], errors="coerce")
df_d["month"] = df_d["date"].dt.to_period("M").dt.to_timestamp()

weight_col = [c for c in df_d.columns if "weight" in c.lower() or "lbs" in c.lower() or "pound" in c.lower()][0]
df_d["lbs"] = pd.to_numeric(df_d[weight_col], errors="coerce")
df_d = df_d.dropna(subset=["date", "lbs"])

monthly_donations = df_d.groupby("month").agg(
    donation_lbs    = ("lbs", "sum"),
    donation_count  = ("lbs", "count"),
    avg_donation    = ("lbs", "mean"),
).reset_index()

print(f"\nMonthly donations: {len(monthly_donations)} months")
print(monthly_donations.head(6))

# ── Merge into one CFB monthly table ──────────────────────────────────────────
df_cfb = monthly_visits.merge(monthly_donations, on="month", how="outer")
df_cfb = df_cfb.sort_values("month").reset_index(drop=True)

# Supply gap: did donations cover food distributed?
df_cfb["supply_gap_lbs"] = df_cfb["donation_lbs"] - df_cfb["food_lbs"]
df_cfb["gap_flag"] = (df_cfb["supply_gap_lbs"] < 0).astype(int)

print(f"\nCFB monthly table: {df_cfb.shape}")
print(f"Gap months: {df_cfb['gap_flag'].sum()} / {len(df_cfb)}")
print(df_cfb.tail(6))

# ── Save ───────────────────────────────────────────────────────────────────────
df_cfb.to_csv("cfb_monthly.csv", index=False)
files.download("cfb_monthly.csv")
print("\n✓ Saved: cfb_monthly.csv")


Monthly visits: 36 months
Date range: 2023-05-01 → 2026-04-01
       month  visits  unique_clients  unique_hh  people_served  food_lbs  \
0 2023-05-01     990             622        618           1645  22214.43   
1 2023-06-01     932             625        622           1565  19528.89   
2 2023-07-01     715             547        545           1198  15421.65   
3 2023-08-01     946             604        603           1588  20147.50   
4 2023-09-01     965             743        741           1572  17442.46   
5 2023-10-01    1044             791        787           1648  22180.20   

   avg_hh_size  lbs_per_person  
0     1.673737       13.504213  
1     1.690987       12.478524  
2     1.679720       12.872830  
3     1.689218       12.687343  
4     1.633161       11.095712  
5     1.586207       13.458859  

Monthly donations: 13 months
       month  donation_lbs  donation_count  avg_donation
0 2023-05-01     11577.070              72    160.792639
1 2023-06-01      8919.990   

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✓ Saved: cfb_monthly.csv


In [3]:
# 1. How many months do you actually have?
print(df_cfb["month"].nunique(), "months")  # expect ~36

# 2. Any months with suspiciously low visits? (data gaps)
print(df_cfb[df_cfb["visits"] < 20])

# 3. Trend — is campus demand growing?
print(df_cfb.groupby(df_cfb["month"].dt.year)[["visits","people_served"]].sum())

36 months
Empty DataFrame
Columns: [month, visits, unique_clients, unique_hh, people_served, food_lbs, avg_hh_size, lbs_per_person, donation_lbs, donation_count, avg_donation, supply_gap_lbs, gap_flag]
Index: []
       visits  people_served
month                       
2023     7362          12137
2024    12676          22028
2025    12392          23216
2026     4238           8863


In [4]:
# Check the household size trend
print(df_cfb.groupby(df_cfb["month"].dt.year)["avg_hh_size"].mean().round(2))

# Check lbs per person trend — is food keeping up with demand?
print(df_cfb.groupby(df_cfb["month"].dt.year)["lbs_per_person"].mean().round(2))

# Gap months breakdown
print(f"Total gap months: {df_cfb['gap_flag'].sum()} / {len(df_cfb)}")
print(df_cfb[df_cfb["gap_flag"]==1][["month","visits","food_lbs","donation_lbs","supply_gap_lbs"]])

# Quick describe
print(df_cfb[["visits","people_served","food_lbs","donation_lbs"]].describe().round(0))

month
2023    1.66
2024    1.77
2025    1.89
2026    2.11
Name: avg_hh_size, dtype: float64
month
2023    12.98
2024    11.52
2025    10.88
2026    10.71
Name: lbs_per_person, dtype: float64
Total gap months: 13 / 36
        month  visits  food_lbs  donation_lbs  supply_gap_lbs
0  2023-05-01     990  22214.43     11577.070      -10637.360
1  2023-06-01     932  19528.89      8919.990      -10608.900
2  2023-07-01     715  15421.65     10932.990       -4488.660
3  2023-08-01     946  20147.50      5693.620      -14453.880
4  2023-09-01     965  17442.46      4462.285      -12980.175
5  2023-10-01    1044  22180.20     10793.730      -11386.470
6  2023-11-01    1016  22888.41     11410.325      -11478.085
7  2023-12-01     754  17643.83     10884.170       -6759.660
8  2024-01-01    1053  21905.62     13642.436       -8263.184
9  2024-02-01     960  20808.00     14739.620       -6068.380
10 2024-03-01    1072  21730.50     12059.200       -9671.300
11 2024-04-01    1071  20625.20     104

In [5]:
# Are the zero-donation months truly zero or just missing data?
print(df_cfb[["month", "donation_lbs", "gap_flag"]].to_string())

# Check raw donation data date range
print(f"Donation data spans: {df_d['date'].min().date()} → {df_d['date'].max().date()}")
print(f"Donation monthly rows: {monthly_donations['month'].nunique()}")

        month  donation_lbs  gap_flag
0  2023-05-01     11577.070         1
1  2023-06-01      8919.990         1
2  2023-07-01     10932.990         1
3  2023-08-01      5693.620         1
4  2023-09-01      4462.285         1
5  2023-10-01     10793.730         1
6  2023-11-01     11410.325         1
7  2023-12-01     10884.170         1
8  2024-01-01     13642.436         1
9  2024-02-01     14739.620         1
10 2024-03-01     12059.200         1
11 2024-04-01     10425.050         1
12 2024-05-01       619.000         1
13 2024-06-01           NaN         0
14 2024-07-01           NaN         0
15 2024-08-01           NaN         0
16 2024-09-01           NaN         0
17 2024-10-01           NaN         0
18 2024-11-01           NaN         0
19 2024-12-01           NaN         0
20 2025-01-01           NaN         0
21 2025-02-01           NaN         0
22 2025-03-01           NaN         0
23 2025-04-01           NaN         0
24 2025-05-01           NaN         0
25 2025-06-0

In [6]:
# Food security pressure index
# Rising = more people competing for less food per person
df_cfb["pressure_index"] = (
    df_cfb["people_served"] / df_cfb["food_lbs"].clip(lower=1)
) * 100   # people per 100 lbs

print(df_cfb.groupby(df_cfb["month"].dt.year)["pressure_index"].mean().round(2))

month
2023    7.74
2024    8.72
2025    9.21
2026    9.35
Name: pressure_index, dtype: float64


In [8]:
# Recalculate gap flag — only where donation data actually exists
# For NaN donation months, flag as "unknown" not "no gap"
df_cfb["gap_flag"] = np.where(
    df_cfb["donation_lbs"].isna(),
    np.nan,                                              # unknown — no donation data
    (df_cfb["supply_gap_lbs"] < 0).astype(float)        # 1=gap, 0=surplus
)

# Separate column for whether donation data exists
df_cfb["has_donation_data"] = df_cfb["donation_lbs"].notna().astype(int)

print(f"Months with donation data : {df_cfb['has_donation_data'].sum()}")
print(f"Gap months (where known)  : {df_cfb['gap_flag'].sum()} / {df_cfb['has_donation_data'].sum()}")
print(f"Months unknown            : {df_cfb['gap_flag'].isna().sum()}")

Months with donation data : 13
Gap months (where known)  : 13.0 / 13
Months unknown            : 23


In [9]:
# Drop gap_flag since it's unreliable after May 2024
# Keep donation cols but flag clearly as partial
df_cfb_clean = df_cfb[[
    "month", "visits", "unique_clients", "unique_hh",
    "people_served", "food_lbs", "avg_hh_size",
    "lbs_per_person", "pressure_index",
    "donation_lbs", "has_donation_data",   # keep but flagged
]].copy()

df_cfb_clean.to_csv("cfb_monthly_clean.csv", index=False)
files.download("cfb_monthly_clean.csv")
print(f"✓ Saved: cfb_monthly_clean.csv — {len(df_cfb_clean)} months")
print(df_cfb_clean[["month","visits","people_served","pressure_index","has_donation_data"]].to_string())

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Saved: cfb_monthly_clean.csv — 36 months
        month  visits  people_served  pressure_index  has_donation_data
0  2023-05-01     990           1645        7.405097                  1
1  2023-06-01     932           1565        8.013768                  1
2  2023-07-01     715           1198        7.768300                  1
3  2023-08-01     946           1588        7.881871                  1
4  2023-09-01     965           1572        9.012490                  1
5  2023-10-01    1044           1648        7.430050                  1
6  2023-11-01    1016           1637        7.152091                  1
7  2023-12-01     754           1284        7.277332                  1
8  2024-01-01    1053           1790        8.171419                  1
9  2024-02-01     960           1580        7.593233                  1
10 2024-03-01    1072           1796        8.264881                  1
11 2024-04-01    1071           1801        8.732037                  1
12 2024-05-01    1164

In [11]:
from google.colab import files
import pandas as pd

print("Upload your external feature CSV (same one used for AFB)...")
up = files.upload()
df_ext_raw = pd.read_csv(list(up.keys())[0])

print(f"Loaded: {df_ext_raw.shape[0]:,} rows x {df_ext_raw.shape[1]} columns")
print(f"Columns: {list(df_ext_raw.columns)}")
print(df_ext_raw.head(3))

Upload your external feature CSV (same one used for AFB)...


Saving data.csv to data.csv
Loaded: 1,977 rows x 23 columns
Columns: ['Date', 'Month', 'holiday_is_stat', 'holiday_is_religious', 'holiday_is_cultural', 'is_holiday', 'is_covid', 'School_In_Session', 'CCB_Dates', 'Exam_Period', 'International_Arrival', 'NLDB', 'CPP', 'OAS', 'CPI All-items', 'CPI Food', 'CPI Shelter', 'Net Migration', 'AISH_TOTAL', 'SINGLE_AISH_TOTAL', 'SINGLE_AISH_PARENT', 'EDMONTON_AISH_CASELOAD', 'Max Temp (°C)']
        Date  Month  holiday_is_stat  holiday_is_religious  \
0  5/31/2026      5                0                     1   
1  5/30/2026      5                0                     1   
2  5/29/2026      5                0                     1   

   holiday_is_cultural  is_holiday  is_covid  School_In_Session  CCB_Dates  \
0                    0           1         0                  1          0   
1                    0           1         0                  1          0   
2                    0           1         0                  1          0   

  

In [14]:
# Merge cfb_monthly_clean + external features
df_ext = df_ext_raw.copy()
df_ext["month"] = pd.to_datetime(df_ext["Date"]).dt.to_period("M").dt.to_timestamp()

# Aggregate to monthly if daily
if df_ext.groupby("month").size().max() > 3:
    num_cols = df_ext.select_dtypes(include="number").columns.tolist()
    df_ext = df_ext.groupby("month")[num_cols].mean().reset_index()

# CFB-specific regressors — school calendar first, then economic
CFB_REGRESSORS = [
    # Tier 1 — school calendar (CFB-specific, expect highest importance)
    "School_In_Session", "Semster_Start", "Exam_Period",
    "Tuition_Payment_Deadline", "Reading_Week",
    "International_Arrival", "Spring_summer_Sem",

    # Tier 2 — economic (confirmed from AFB + RDFB)
    "EDMONTON_AISH_CASELOAD", "SINGLE_AISH_TOTAL",
    "CPI Food", "CPI All-items", "CPI Shelter",
    "NLDB", "Unemployment_Rate",

    # Tier 3 — other
    "holiday_is_stat", "holiday_is_religious",
    "Tax_Season", "is_covid",
]
CFB_REGRESSORS = [r for r in CFB_REGRESSORS if r in df_ext.columns]

df_cfb_model = df_cfb_clean.merge(
    df_ext[["month"] + CFB_REGRESSORS], on="month", how="left"
)

# Fix AISH comma strings
for col in ["EDMONTON_AISH_CASELOAD", "SINGLE_AISH_TOTAL"]:
    if col in df_cfb_model.columns:
        df_cfb_model[col] = pd.to_numeric(
            df_cfb_model[col].astype(str).str.replace(",", ""), errors="coerce"
        )

print(f"CFB model dataset: {df_cfb_model.shape}")
print(f"Regressors available: {CFB_REGRESSORS}")
print(df_cfb_model[["month","visits","people_served"] + CFB_REGRESSORS[:4]].tail(6))

CFB model dataset: (36, 23)
Regressors available: ['School_In_Session', 'Exam_Period', 'International_Arrival', 'EDMONTON_AISH_CASELOAD', 'SINGLE_AISH_TOTAL', 'CPI Food', 'CPI All-items', 'CPI Shelter', 'NLDB', 'holiday_is_stat', 'holiday_is_religious', 'is_covid']
        month  visits  people_served  School_In_Session  Exam_Period  \
30 2025-11-01    1069           2057           1.000000     0.000000   
31 2025-12-01     767           1590           0.741935     0.870968   
32 2026-01-01    1133           2261           0.935484     0.000000   
33 2026-02-01     958           2023           0.928571     0.000000   
34 2026-03-01    1140           2389           0.935484     0.000000   
35 2026-04-01    1007           2190           0.933333     0.700000   

    International_Arrival  EDMONTON_AISH_CASELOAD  
30               0.000000                 38152.0  
31               0.387097                 38196.0  
32               0.322581                 38240.0  
33               0.00

In [22]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df_cfb_model[CFB_REGRESSORS] = scaler.fit_transform(
    df_cfb_model[CFB_REGRESSORS].fillna(0)
)

corrs = (
    df_cfb_model[["visits"] + CFB_REGRESSORS]
    .corr()["visits"]
    .drop("visits")
    .abs()
    .sort_values(ascending=False)
)
print(corrs.round(3).to_string())

# Keep only features with |r| > 0.15 for 36-month dataset
FINAL_REGRESSORS = corrs[corrs > 0.15].index.tolist()
print(f"\nFinal regressors for CFB model: {FINAL_REGRESSORS}")

Exam_Period               0.482
holiday_is_stat           0.396
School_In_Session         0.362
International_Arrival     0.309
SINGLE_AISH_TOTAL         0.291
CPI All-items             0.283
holiday_is_religious      0.283
CPI Shelter               0.243
CPI Food                  0.119
EDMONTON_AISH_CASELOAD    0.048
NLDB                      0.010
is_covid                    NaN

Final regressors for CFB model: ['Exam_Period', 'holiday_is_stat', 'School_In_Session', 'International_Arrival', 'SINGLE_AISH_TOTAL', 'CPI All-items', 'holiday_is_religious', 'CPI Shelter']


In [19]:
# ── Rebuild with minimal regressors ───────────────────────────────────────────
# Only top 2 from correlation — let Prophet seasonality handle the rest
SIMPLE_REGRESSORS = ["Exam_Period", "School_In_Session"]

def to_prophet_cfb(df, target):
    return (
        df[["month", target] + SIMPLE_REGRESSORS]
        .rename(columns={"month": "ds", target: "y"})
        .dropna()
    )

df_visits = to_prophet_cfb(df_cfb_model, "visits")
df_people = to_prophet_cfb(df_cfb_model, "people_served")

# ── Unscale regressors — Prophet needs original 0/1 flag values ───────────────
# Exam_Period and School_In_Session are originally 0–1 proportions
# Re-extract from raw merged data before scaling was applied
df_visits["Exam_Period"]       = df_cfb_model["Exam_Period"].values
df_visits["School_In_Session"] = df_cfb_model["School_In_Session"].values
df_people["Exam_Period"]       = df_cfb_model["Exam_Period"].values
df_people["School_In_Session"] = df_cfb_model["School_In_Session"].values

print(f"Regressor ranges (should be 0–1):")
print(f"  Exam_Period      : {df_visits['Exam_Period'].min():.2f} – {df_visits['Exam_Period'].max():.2f}")
print(f"  School_In_Session: {df_visits['School_In_Session'].min():.2f} – {df_visits['School_In_Session'].max():.2f}")

# ── Rebuild Prophet ────────────────────────────────────────────────────────────
def build_cfb_prophet(cp=0.3):
    m = Prophet(
        yearly_seasonality      = True,
        weekly_seasonality      = False,
        daily_seasonality       = False,
        seasonality_mode        = "additive",
        changepoint_prior_scale = cp,
        interval_width          = 0.80,
    )
    for r in SIMPLE_REGRESSORS:
        m.add_regressor(r)
    return m

print("\nFitting visits model...")
model_visits = build_cfb_prophet()
model_visits.fit(df_visits)

print("Fitting people_served model...")
model_people = build_cfb_prophet()
model_people.fit(df_people)

# ── In-sample check ────────────────────────────────────────────────────────────
for label, model, df_p in [
    ("visits",        model_visits, df_visits),
    ("people_served", model_people, df_people),
]:
    fit    = model.predict(df_p)
    y_true = df_p["y"].values
    y_pred = np.maximum(fit["yhat"].values, 0)
    mae    = mean_absolute_error(y_true, y_pred)
    mape   = np.mean(np.abs((y_true - y_pred) / y_true.clip(1))) * 100
    print(f"\n{label} in-sample:  MAE={mae:,.0f}  MAPE={mape:.1f}%")

Regressor ranges (should be 0–1):
  Exam_Period      : -0.68 – 2.06
  School_In_Session: -2.93 – 0.72

Fitting visits model...
Fitting people_served model...

visits in-sample:  MAE=14  MAPE=1.4%

people_served in-sample:  MAE=23  MAPE=1.2%


In [17]:
# 36 months total — use 18 months initial, retrain every 3, horizon 3
print("Cross-validating visits model...")
cv_visits = cross_validation(
    model_visits,
    initial  = "548 days",    # ~18 months
    period   = "90 days",     # retrain every 3 months
    horizon  = "90 days",     # 3-month forecast
    parallel = None,
)
pm_visits = performance_metrics(cv_visits)

print("Cross-validating people_served model...")
cv_people = cross_validation(
    model_people,
    initial  = "548 days",
    period   = "90 days",
    horizon  = "90 days",
    parallel = None,
)
pm_people = performance_metrics(cv_people)

print(f"\nVisits model CV:")
print(f"  MAE  : {pm_visits['mae'].mean():,.0f}")
print(f"  MAPE : {pm_visits['mape'].mean()*100:.1f}%")

print(f"\nPeople served model CV:")
print(f"  MAE  : {pm_people['mae'].mean():,.0f}")
print(f"  MAPE : {pm_people['mape'].mean()*100:.1f}%")

INFO:prophet:Making 5 forecasts with cutoffs between 2025-01-06 00:00:00 and 2026-01-01 00:00:00


Cross-validating visits model...


  0%|          | 0/5 [00:00<?, ?it/s]

INFO:prophet:n_changepoints greater than number of observations. Using 15.
INFO:prophet:n_changepoints greater than number of observations. Using 18.
INFO:prophet:n_changepoints greater than number of observations. Using 20.
INFO:prophet:n_changepoints greater than number of observations. Using 23.
INFO:prophet:Making 5 forecasts with cutoffs between 2025-01-06 00:00:00 and 2026-01-01 00:00:00


Cross-validating people_served model...


  0%|          | 0/5 [00:00<?, ?it/s]

INFO:prophet:n_changepoints greater than number of observations. Using 15.
INFO:prophet:n_changepoints greater than number of observations. Using 18.
INFO:prophet:n_changepoints greater than number of observations. Using 20.
INFO:prophet:n_changepoints greater than number of observations. Using 23.



Visits model CV:
  MAE  : 507
  MAPE : 47.9%

People served model CV:
  MAE  : 1,328
  MAPE : 66.9%


In [23]:
# ── Pull original unscaled values from raw merged data ─────────────────────────
# Re-merge cfb_clean with external features WITHOUT scaling this time
df_ext_unscaled = df_ext_raw.copy()
df_ext_unscaled["Date"] = pd.to_datetime(df_ext_unscaled["Date"])
df_ext_unscaled["month"]  = df_ext_unscaled["Date"].dt.to_period("M").dt.to_timestamp()

if df_ext_unscaled.groupby("month").size().max() > 3:
    num_cols = df_ext_unscaled.select_dtypes(include="number").columns.tolist()
    df_ext_unscaled = df_ext_unscaled.groupby("month")[num_cols].mean().reset_index()

# Merge onto clean CFB data — no scaling
df_cfb_prophet = df_cfb_clean.merge(
    df_ext_unscaled[["month", "Exam_Period", "School_In_Session"]],
    on="month", how="left"
)

print("Regressor ranges (unscaled):")
print(f"  Exam_Period      : {df_cfb_prophet['Exam_Period'].min():.2f} – {df_cfb_prophet['Exam_Period'].max():.2f}")
print(f"  School_In_Session: {df_cfb_prophet['School_In_Session'].min():.2f} – {df_cfb_prophet['School_In_Session'].max():.2f}")
print(df_cfb_prophet[["month","visits","Exam_Period","School_In_Session"]].tail(6))

Regressor ranges (unscaled):
  Exam_Period      : 0.00 – 0.87
  School_In_Session: 0.74 – 1.00
        month  visits  Exam_Period  School_In_Session
30 2025-11-01    1069     0.000000           1.000000
31 2025-12-01     767     0.870968           0.741935
32 2026-01-01    1133     0.000000           0.935484
33 2026-02-01     958     0.000000           0.928571
34 2026-03-01    1140     0.000000           0.935484
35 2026-04-01    1007     0.700000           0.933333


In [24]:
SIMPLE_REGRESSORS = ["Exam_Period", "School_In_Session"]

df_visits = (
    df_cfb_prophet[["month", "visits"] + SIMPLE_REGRESSORS]
    .rename(columns={"month": "ds", "visits": "y"})
    .dropna()
)
df_people = (
    df_cfb_prophet[["month", "people_served"] + SIMPLE_REGRESSORS]
    .rename(columns={"month": "ds", "people_served": "y"})
    .dropna()
)

print(f"Visits df : {len(df_visits)} rows")
print(f"People df : {len(df_people)} rows")

# Refit
model_visits = build_cfb_prophet(cp=0.3)
model_visits.fit(df_visits)

model_people = build_cfb_prophet(cp=0.3)
model_people.fit(df_people)

# In-sample
for label, model, df_p in [
    ("visits",        model_visits, df_visits),
    ("people_served", model_people, df_people),
]:
    fit    = model.predict(df_p)
    y_true = df_p["y"].values
    y_pred = np.maximum(fit["yhat"].values, 0)
    mae    = mean_absolute_error(y_true, y_pred)
    mape   = np.mean(np.abs((y_true - y_pred) / y_true.clip(1))) * 100
    print(f"{label}: MAE={mae:,.0f}  MAPE={mape:.1f}%")

# CV
print("\nCross-validating...")
cv_visits = cross_validation(
    model_visits, initial="548 days", period="90 days",
    horizon="90 days", parallel=None
)
cv_people = cross_validation(
    model_people, initial="548 days", period="90 days",
    horizon="90 days", parallel=None
)

pm_v = performance_metrics(cv_visits)
pm_p = performance_metrics(cv_people)

print(f"\nVisits CV:  MAE={pm_v['mae'].mean():,.0f}  MAPE={pm_v['mape'].mean()*100:.1f}%")
print(f"People CV:  MAE={pm_p['mae'].mean():,.0f}  MAPE={pm_p['mape'].mean()*100:.1f}%")

Visits df : 36 rows
People df : 36 rows


INFO:prophet:Making 5 forecasts with cutoffs between 2025-01-06 00:00:00 and 2026-01-01 00:00:00


visits: MAE=15  MAPE=1.5%
people_served: MAE=23  MAPE=1.2%

Cross-validating...


  0%|          | 0/5 [00:00<?, ?it/s]

INFO:prophet:n_changepoints greater than number of observations. Using 15.
INFO:prophet:n_changepoints greater than number of observations. Using 18.
INFO:prophet:n_changepoints greater than number of observations. Using 20.
INFO:prophet:n_changepoints greater than number of observations. Using 23.
INFO:prophet:Making 5 forecasts with cutoffs between 2025-01-06 00:00:00 and 2026-01-01 00:00:00


  0%|          | 0/5 [00:00<?, ?it/s]

INFO:prophet:n_changepoints greater than number of observations. Using 15.
INFO:prophet:n_changepoints greater than number of observations. Using 18.
INFO:prophet:n_changepoints greater than number of observations. Using 20.
INFO:prophet:n_changepoints greater than number of observations. Using 23.



Visits CV:  MAE=144  MAPE=13.8%
People CV:  MAE=990  MAPE=52.3%


In [25]:
import json

N_MONTHS = 12
last_month    = df_cfb_prophet["month"].max()
future_months = pd.date_range(
    start   = last_month + pd.offsets.MonthBegin(1),
    periods = N_MONTHS,
    freq    = "MS",
)

# ── Build future regressors from prior year calendar pattern ───────────────────
all_future = pd.DataFrame({"month": future_months})
for col in SIMPLE_REGRESSORS:
    for i, row in all_future.iterrows():
        prior = row["month"] - pd.DateOffset(years=1)
        prior_val = df_cfb_prophet[df_cfb_prophet["month"] == prior][col]
        if len(prior_val) > 0:
            all_future.loc[i, col] = prior_val.values[0]
        else:
            month_num = row["month"].month
            mean_val  = df_cfb_prophet[
                df_cfb_prophet["month"].dt.month == month_num
            ][col].mean()
            all_future.loc[i, col] = mean_val

all_future["ds"] = all_future["month"]

# ── Combine historical + future ────────────────────────────────────────────────
df_future = pd.concat([
    df_visits[["ds"] + SIMPLE_REGRESSORS],
    all_future[["ds"] + SIMPLE_REGRESSORS],
], ignore_index=True).fillna(0)

fc_visits = model_visits.predict(df_future)
fv = fc_visits[fc_visits["ds"] >= future_months[0]].copy()
fv["yhat"]       = np.maximum(fv["yhat"],       0)
fv["yhat_lower"] = np.maximum(fv["yhat_lower"], 0)
fv["yhat_upper"] = np.maximum(fv["yhat_upper"], 0)

# ── Print forecast table ───────────────────────────────────────────────────────
print("=" * 65)
print(f"  {'Month':<12} {'Visits':>8} {'80% Low':>9} {'80% High':>10}")
print("=" * 65)
for _, row in fv.iterrows():
    print(f"  {row['ds'].strftime('%b %Y'):<12} "
          f"{row['yhat']:>8,.0f} "
          f"{row['yhat_lower']:>9,.0f} "
          f"{row['yhat_upper']:>10,.0f}")
print("=" * 65)
print(f"  Mean forecast: {fv['yhat'].mean():,.0f} visits/month")
print(f"  Historical mean: {df_visits['y'].mean():,.0f} visits/month")

# ── Export JSON for dashboard ──────────────────────────────────────────────────
# Last 2 years historical
cutoff   = df_cfb_prophet["month"].max() - pd.DateOffset(years=2)
df_hist  = df_cfb_prophet[df_cfb_prophet["month"] >= cutoff].copy()
fit_hist = model_visits.predict(
    df_visits[df_visits["ds"] >= cutoff][["ds"] + SIMPLE_REGRESSORS]
)
fit_hist["yhat"] = np.maximum(fit_hist["yhat"], 0)

historical = [
    {
        "month":  row["month"].strftime("%Y-%m-%d"),
        "label":  row["month"].strftime("%b %Y"),
        "actual": int(row["visits"]),
        "fitted": round(float(
            fit_hist[fit_hist["ds"] == row["month"]]["yhat"].values[0]
        ), 1) if len(fit_hist[fit_hist["ds"] == row["month"]]) > 0 else None,
        "people_served": int(row["people_served"]) if pd.notna(row["people_served"]) else None,
        "lbs_per_person": round(float(row["lbs_per_person"]), 2) if pd.notna(row["lbs_per_person"]) else None,
        "pressure_index": round(float(row["pressure_index"]), 2) if pd.notna(row["pressure_index"]) else None,
    }
    for _, row in df_hist.iterrows()
]

forecast_out = [
    {
        "month":    row["ds"].strftime("%Y-%m-%d"),
        "label":    row["ds"].strftime("%b %Y"),
        "forecast": round(float(row["yhat"]), 1),
        "lower_80": round(float(row["yhat_lower"]), 1),
        "upper_80": round(float(row["yhat_upper"]), 1),
    }
    for _, row in fv.iterrows()
]

output = {
    "model":      "CFB Prophet",
    "generated":  pd.Timestamp.now().strftime("%Y-%m-%d"),
    "units":      "visits per month",
    "accuracy": {
        "cv_mape":     13.8,
        "cv_mae":      144,
        "in_sample_mape": 1.4,
    },
    "top_drivers": [
        {"feature": "Exam_Period",       "correlation": 0.482},
        {"feature": "School_In_Session", "correlation": 0.362},
    ],
    "trends": {
        "avg_hh_size_2023": 1.66,
        "avg_hh_size_2026": 2.11,
        "lbs_per_person_2023": 12.98,
        "lbs_per_person_2026": 10.71,
        "pressure_index_2023": 7.74,
        "pressure_index_2026": 9.35,
    },
    "historical": historical,
    "forecast":   forecast_out,
}

with open("cfb_forecast.json", "w") as f:
    json.dump(output, f, indent=2)

print(f"\n✓ Saved: cfb_forecast.json")
print(f"  {len(historical)} historical months + {len(forecast_out)} forecast months")
files.download("cfb_forecast.json")

  Month          Visits   80% Low   80% High
  May 2026        1,103     1,078      1,127
  Jun 2026        1,013       988      1,036
  Jul 2026          853       830        878
  Aug 2026          986       959      1,011
  Sep 2026        1,055     1,024      1,083
  Oct 2026        1,109     1,079      1,139
  Nov 2026        1,062     1,027      1,097
  Dec 2026          729       691        765
  Jan 2027        1,111     1,066      1,151
  Feb 2027          945       897        987
  Mar 2027        1,145     1,091      1,195
  Apr 2027        1,005       948      1,060
  Mean forecast: 1,010 visits/month
  Historical mean: 1,019 visits/month

✓ Saved: cfb_forecast.json
  25 historical months + 12 forecast months


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [21]:
N_MONTHS = 12
last_month    = df_cfb_model["month"].max()
future_months = pd.date_range(
    start   = last_month + pd.offsets.MonthBegin(1),
    periods = N_MONTHS,
    freq    = "MS",
)

# ── Build future regressors from RAW external data ─────────────────────────────
# Don't forward-fill scaled values — get actual calendar values for future months
df_ext_future = df_ext_raw.copy()
df_ext_future["Date"] = pd.to_datetime(df_ext_future["Date"])
df_ext_future["month"]  = df_ext_future["Date"].dt.to_period("M").dt.to_timestamp()

if df_ext_future.groupby("month").size().max() > 3:
    num_cols = df_ext_future.select_dtypes(include="number").columns.tolist()
    df_ext_future = df_ext_future.groupby("month")[num_cols].mean().reset_index()

# Get future months from external data if available, else forward-fill
future_regs = df_ext_future[
    df_ext_future["month"].isin(future_months)
][["month", "Exam_Period", "School_In_Session"]].copy()

if len(future_regs) < N_MONTHS:
    # External data doesn't cover future — use last known calendar pattern
    # School calendar repeats yearly so use same month from prior year
    print(f"External data covers {len(future_regs)}/{N_MONTHS} future months")
    print("Filling remaining months using prior year calendar pattern...")

    all_future = pd.DataFrame({"month": future_months})
    all_future = all_future.merge(future_regs, on="month", how="left")

    for col in ["Exam_Period", "School_In_Session"]:
        for i, row in all_future[all_future[col].isna()].iterrows():
            # Same month, prior year
            prior = row["month"] - pd.DateOffset(years=1)
            prior_val = df_cfb_model[
                df_cfb_model["month"] == prior
            ][col]
            if len(prior_val) > 0:
                all_future.loc[i, col] = prior_val.values[0]
            else:
                # Fallback: use mean for that calendar month
                month_num = row["month"].month
                mean_val = df_cfb_model[
                    df_cfb_model["month"].dt.month == month_num
                ][col].mean()
                all_future.loc[i, col] = mean_val
    future_regs = all_future

future_regs["ds"] = future_regs["month"]

# ── Combine historical + future ────────────────────────────────────────────────
df_future = pd.concat([
    df_visits[["ds"] + SIMPLE_REGRESSORS],
    future_regs[["ds"] + SIMPLE_REGRESSORS],
], ignore_index=True).fillna(0)

fc_visits = model_visits.predict(df_future)
fc_people = model_people.predict(df_future)

fv = fc_visits[fc_visits["ds"] >= future_months[0]].copy()
fp = fc_people[fc_people["ds"] >= future_months[0]].copy()

for df_f in [fv, fp]:
    df_f["yhat"]       = np.maximum(df_f["yhat"],       0)
    df_f["yhat_lower"] = np.maximum(df_f["yhat_lower"], 0)
    df_f["yhat_upper"] = np.maximum(df_f["yhat_upper"], 0)

print("=" * 80)
print(f"  {'Month':<12} {'Visits':>8} {'V low':>8} {'V high':>8} "
      f"{'People':>8} {'P low':>8} {'P high':>8}")
print("=" * 80)
for (_, rv), (_, rp) in zip(fv.iterrows(), fp.iterrows()):
    print(f"  {rv['ds'].strftime('%b %Y'):<12} "
          f"{rv['yhat']:>8,.0f} {rv['yhat_lower']:>8,.0f} {rv['yhat_upper']:>8,.0f} "
          f"{rp['yhat']:>8,.0f} {rp['yhat_lower']:>8,.0f} {rp['yhat_upper']:>8,.0f}")
print("=" * 80)

External data covers 1/12 future months
Filling remaining months using prior year calendar pattern...
  Month          Visits    V low   V high   People    P low   P high
  May 2026        5,267    5,244    5,291   11,934   11,891   11,973
  Jun 2026        1,004      981    1,029    2,278    2,234    2,320
  Jul 2026          844      819      869    2,071    2,018    2,118
  Aug 2026          975      948    1,000    2,326    2,264    2,383
  Sep 2026        1,044    1,016    1,072    2,463    2,390    2,536
  Oct 2026        1,098    1,066    1,131    2,577    2,480    2,666
  Nov 2026        1,051    1,016    1,087    2,516    2,405    2,620
  Dec 2026          720      678      760    1,995    1,860    2,123
  Jan 2027        1,105    1,061    1,154    2,747    2,587    2,904
  Feb 2027          937      884      987    2,421    2,235    2,597
  Mar 2027        1,132    1,075    1,192    2,816    2,601    3,023
  Apr 2027          994      930    1,060    2,691    2,456    2,927
